# 02 - Formulation spike (T4.2.1)

Compares **A** (single MIP with exact allergen sequencing) and **B** (lot-sizing MIP + CP-SAT sequencing per week) on a 4 SKU x 8 week toy and the full 12 x 13 instance, synthetic demand (seed 11). All plans are scored by the same simulator-based cost evaluator. Decision: `docs/decisions/ADR-003-formulation.md`.

Re-run the solves (about 10 min): `python scripts/spike_formulation.py`. This notebook only reads the saved results.

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(root / 'results/logs/spike_formulation.csv')
cols = ['instance','approach','solver','status','solve_s','gap','mip_obj','eval_cost','changeover_h','lost','expired','fill_rate','capacity_breach_weeks']
df[cols].round(3)

,instance,approach,solver,status,solve_s,gap,mip_obj,eval_cost,changeover_h,lost,expired,fill_rate,capacity_breach_weeks
0,toy_4x8,A_arc,SCIP,OPTIMAL,0.584,0.019,2426048.250,2426048.250,43.0,2096.000,0.0,0.984,0.0
1,toy_4x8,A_arc,HIGHS,OPTIMAL,0.708,0.000,2379990.750,2379990.750,42.0,213.000,0.0,0.998,0.0
2,toy_4x8,B_decomp,SCIP,OPTIMAL,0.164,0.000,2391732.167,2381265.500,38.0,213.000,0.0,0.998,0.0
3,toy_4x8,B_decomp,HIGHS,OPTIMAL,0.222,0.000,2391508.604,2382441.938,41.5,213.000,0.0,0.998,0.0
4,toy_4x8,A_arc,CBC,OPTIMAL,1.407,0.002,2383161.000,2383161.000,50.5,213.000,0.0,0.998,0.0
5,full_12x13,A_arc,SCIP,FEASIBLE,119.906,0.028,4050329.267,4050329.267,214.0,5018.074,0.0,0.976,0.0
6,full_12x13,A_arc,HIGHS,99,120.321,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,full_12x13,B_decomp,SCIP,OPTIMAL,3.375,0.012,4132692.493,4110801.584,155.5,8484.582,0.0,0.960,0.0
8,full_12x13,B_decomp,HIGHS,99,120.064,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,full_12x13,A_arc,CBC,FEASIBLE,119.904,0.037,4086479.807,4086479.807,205.0,6847.954,0.0,0.968,0.0


## Does the optimiser's accounting match the simulator?
For approach A the MIP objective should equal the simulator-evaluated cost of the same plan.

In [2]:
a = df[(df.approach == 'A_arc') & df.eval_cost.notna()]
(a.assign(diff_pct=100 * (a.eval_cost - a.mip_obj) / a.mip_obj)[['instance','solver','mip_obj','eval_cost','diff_pct']]).round(4)

,instance,solver,mip_obj,eval_cost,diff_pct
0,toy_4x8,SCIP,2.426048e+06,2.426048e+06,0.0
1,toy_4x8,HIGHS,2.379991e+06,2.379991e+06,0.0
4,toy_4x8,CBC,2.383161e+06,2.383161e+06,0.0
5,full_12x13,SCIP,4.050329e+06,4.050329e+06,0.0
9,full_12x13,CBC,4.086480e+06,4.086480e+06,0.0


## Full instance: cost vs solve time

In [3]:
f = df[(df.instance == 'full_12x13') & df.eval_cost.notna()].copy()
best = f.eval_cost.min()
f['cost_vs_best_pct'] = 100 * (f.eval_cost / best - 1)
f[['approach','solver','solve_s','gap','cost_vs_best_pct','changeover_h','lost']].round(2)

,approach,solver,solve_s,gap,cost_vs_best_pct,changeover_h,lost
5,A_arc,SCIP,119.91,0.03,0.00,214.0,5018.07
7,B_decomp,SCIP,3.37,0.01,1.49,155.5,8484.58
9,A_arc,CBC,119.90,0.04,0.89,205.0,6847.95


## Reading
- B (SCIP) solves the 12 x 13 instance in ~3 s, within ~1.5% of A's best incumbent, which needed the full 120 s limit (2.8% gap).
- B's per-run changeover estimate is pessimistic: fewer changeover hours but more lost cases than A. Follow-up in T4.2.3.
- HiGHS (via pywraplp) found no solution in 120 s on either full model, so SCIP is used.
- Runtime implication: a 52-week rolling replay costs ~3 min with B versus ~1.7 h with A; the Monte Carlo layer needs parallel processes (ADR-003).